# Netflix 电影与电视节目 · 数据分析报告

---

## 小组分工（两人组）

- **成员 A · 数据科学家**：数据加载、预处理（清洗 / 缺失值处理 / 异常值检测）、描述性统计、分组聚合分析、JSON 数据导出
- **成员 B · 前端工程师**：pyecharts 交互式图表开发、网页前端 `index.html` 搭建、可视化呈现、答辩演示

---

## 1. 数据集背景

| 项目 | 内容 |
| --- | --- |
| 数据集 | Netflix Titles Dataset |
| 记录数 | 8,807 部作品 |
| 字段数 | 12 列（show_id, type, title, director, cast, country, date_added, release_year, rating, duration, listed_in, description）|
| 来源 | Kaggle 公开数据集 |

## 2. 分析目标
1. 电影与电视节目的数量与占比分布
2. 内容随时间的增长趋势
3. 主要制片国家的产出排行
4. 热门流派 Top10
5. 年龄评级分布
6. 电影时长分布 & 电视节目季数分布
7. 发行年份 × 评级 交叉热力图

---

In [ ]:
import pandas as pd
import numpy as np
import json
from collections import Counter
from pyecharts import options as opts
from pyecharts.charts import Bar, Pie, Line, WordCloud, HeatMap, Funnel, Page
from pyecharts.globals import ThemeType, CurrentConfig

CurrentConfig.ONLINE_HOST = "https://cdn.jsdelivr.net/npm/echarts@5.4.3/dist/"

# -------- 数据加载 --------
df = pd.read_csv('netflix_titles.csv')
df

In [ ]:
# -------- 数据预处理 --------
# 缺失值填充
for col in ['director', 'cast', 'country']:
    df[col] = df[col].fillna('Unknown')
df['date_added'] = df['date_added'].fillna(df['date_added'].mode()[0])
df['rating'] = df['rating'].fillna(df['rating'].mode()[0])
df['duration'] = df['duration'].fillna(df['duration'].mode()[0])

# 提取添加年份
df['year_added'] = df['date_added'].str.extract(r'(\d{4})').astype(int)

# 拆分多值字段
def split_field(s):
    if pd.isna(s) or s == 'Unknown':
        return []
    return [x.strip() for x in str(s).split(',') if x.strip()]

df['genres_list'] = df['listed_in'].apply(split_field)
df['countries_list'] = df['country'].apply(split_field)
df['cast_list'] = df['cast'].apply(split_field)
df['duration_num'] = df['duration'].str.extract(r'(\d+)').astype(float).astype(int)

print('缺失值总数:', df.isnull().sum().sum())
df

In [ ]:
# -------- 描述性统计 --------
type_dist = df['type'].value_counts().reset_index()
type_dist.columns = ['类型', '数量']
type_dist['占比(%)'] = (type_dist['数量'] / type_dist['数量'].sum() * 100).round(2)
type_dist

In [ ]:
# 按年份统计新增内容
year_added = df.groupby(['year_added', 'type']).size().unstack().fillna(0).astype(int)
year_added['总计'] = year_added.sum(axis=1)
year_added.sort_index()

In [ ]:
# 国家 Top15 & 流派 Top10 & 评级 Top10
country_count = pd.DataFrame(
    Counter(c for cs in df['countries_list'] for c in cs).most_common(15),
    columns=['国家', '数量']
)
genre_count = pd.DataFrame(
    Counter(g for gs in df['genres_list'] for g in gs).most_common(10),
    columns=['流派', '数量']
)
rating_dist = df['rating'].value_counts().head(10).reset_index()
rating_dist.columns = ['评级', '数量']

print('=== 制片国家 Top5 ===')
display(country_count.head())
print('\n=== 热门流派 Top5 ===')
display(genre_count.head())
print('\n=== 年龄评级 Top5 ===')
display(rating_dist.head())

In [ ]:
# 电影时长 & 电视节目季数的描述性统计
movie_dur = df[df['type'] == 'Movie']['duration_num']
show_seasons = df[df['type'] == 'TV Show']['duration_num']
print('=== 电影时长(分钟) ===')
print(f'均值: {movie_dur.mean():.2f}  |  中位数: {movie_dur.median():.2f}  |  最小: {movie_dur.min()}  |  最大: {movie_dur.max()}')
print(f'\n=== 电视节目季数 ===')
print(f'均值: {show_seasons.mean():.2f}  |  中位数: {show_seasons.median():.2f}  |  最小: {show_seasons.min()}  |  最大: {show_seasons.max()}')

---

## 3. 使用 pyecharts 绘制可视化图表

以下仅展示关键代码片段（更多完整图表请查看 `index.html` 网页前端）。

In [ ]:
# 图1：电影 vs 电视节目 环形图
data_pie = [list(z) for z in zip(type_dist['类型'].tolist(), type_dist['数量'].tolist())]
c1 = (
    Pie(init_opts=opts.InitOpts(theme=ThemeType.DARK, width='100%', height='500px'))
    .add(series_name='数量分布', data_pair=data_pie, radius=['40%', '70%'],
         label_opts=opts.LabelOpts(formatter='{b}: {c} ({d}%)', color='#fff'))
    .set_colors(['#E50914', '#FFD700'])
    .set_global_opts(title_opts=opts.TitleOpts(title='Movie vs TV Show', pos_left='center'))
)
c1

In [ ]:
# 图2：新增内容趋势折线图
years = [int(y) for y in year_added.index.tolist()]
movies = year_added.get('Movie', pd.Series(0, index=year_added.index)).tolist()
shows = year_added.get('TV Show', pd.Series(0, index=year_added.index)).tolist()

c2 = (
    Line(init_opts=opts.InitOpts(theme=ThemeType.DARK, width='100%', height='550px'))
    .add_xaxis(years)
    .add_yaxis('电影', movies, is_smooth=True, linestyle_opts=opts.LineStyleOpts(width=4, color='#E50914'))
    .add_yaxis('电视节目', shows, is_smooth=True, linestyle_opts=opts.LineStyleOpts(width=4, color='#FFD700'))
    .set_global_opts(title_opts=opts.TitleOpts(title='Content added per year', pos_left='center'),
                     tooltip_opts=opts.TooltipOpts(trigger='axis', axis_pointer_type='cross'))
)
c2

In [ ]:
# 图3：国家 Top15 柱状图
countries_r = country_count['国家'].tolist()[::-1]
counts_r = country_count['数量'].tolist()[::-1]

c3 = (
    Bar(init_opts=opts.InitOpts(theme=ThemeType.DARK, width='100%', height='600px'))
    .add_xaxis(countries_r)
    .add_yaxis('作品数量', counts_r, category_gap='40%',
               label_opts=opts.LabelOpts(position='right', color='#fff'))
    .reversal_axis()
    .set_global_opts(title_opts=opts.TitleOpts(title='Top 15 Countries', pos_left='center'))
)
c3

In [ ]:
# 图4：评级漏斗图
data_funnel = [list(z) for z in zip(rating_dist['评级'].tolist(), rating_dist['数量'].tolist())]
c5 = (
    Funnel(init_opts=opts.InitOpts(theme=ThemeType.DARK, width='100%', height='550px'))
    .add(series_name='评级', data_pair=data_funnel, sort_='descending', gap=2,
         label_opts=opts.LabelOpts(position='inside', formatter='{b}: {c}', color='#fff'))
    .set_global_opts(title_opts=opts.TitleOpts(title='Rating Distribution (Funnel)', pos_left='center'))
)
c5

In [ ]:
# 导出 JSON 数据给前端使用
overview = {
    'total': int(df.shape[0]),
    'movies': int(df[df['type'] == 'Movie'].shape[0]),
    'tvshows': int(df[df['type'] == 'TV Show'].shape[0]),
    'countries': len(set(c for cs in df['countries_list'] for c in cs)),
    'genres': len(set(g for gs in df['genres_list'] for g in gs)),
    'directors': int(df[df['director'] != 'Unknown']['director'].nunique()),
}

year_export = [{'year': int(idx),
                'Movie': int(row.get('Movie', 0)),
                'TV Show': int(row.get('TV Show', 0)),
                'total': int(row['总计'])}
               for idx, row in year_added.sort_index().iterrows()]

dur_stats = {'mean': float(movie_dur.mean()), 'median': float(movie_dur.median()),
             'min': float(movie_dur.min()), 'max': float(movie_dur.max()), 'count': int(len(movie_dur))}

export = {
    'overview': overview,
    'type_dist': type_dist.to_dict(orient='records'),
    'year_trend': year_export,
    'countries': country_count.to_dict(orient='records'),
    'genres': genre_count.to_dict(orient='records'),
    'ratings': rating_dist.to_dict(orient='records'),
    'duration_stats': dur_stats,
    # 新增：交叉分析数据
    'genre_type_breakdown': genre_type_data,
    'season_distribution': season_df.to_dict('records') if 'season_df' in dir() else [],
    'heatmap_data': heatmap_real if 'heatmap_real' in dir() else [],
    'heatmap_meta': {'years': heatmap_years, 'ratings': heatmap_ratings} if 'heatmap_years' in dir() else {},
    'duration_histogram': duration_histogram if 'duration_histogram' in dir() else [],
}

with open('data/analysis_data.json', 'w', encoding='utf-8') as f:
    json.dump(export, f, ensure_ascii=False, indent=2)
print('JSON 数据已导出到 data/analysis_data.json')
print('概览:', overview)

---

## 4. 分析结论

| 发现 | 结论 |
| --- | --- |
| **电影占主导** | 电影 6,131 部 (~70%) vs 电视节目 2,676 部 (~30%) |
| **增长期集中** | 2016-2019 年内容爆发式增长，2019 年峰值约 2,016 部 |
| **美国核心** | 美国 3,690 部遥遥领先，印度、英、加紧随 |
| **国际电影最热门** | International Movies / Dramas / Comedies 三大主流 |
| **单季剧居多** | 多数 TV Show 仅 1 季，Netflix 策略偏向新剧快速迭代 |
| **成人向为主** | TV-MA 评级 3,211 部，成年观众内容占主导 |

### 业务建议
1. 加强非英语地区（亚洲 / 欧洲）原创内容，匹配全球用户文化需求
2. 重点打造多季爆款 IP，提升订阅粘性
3. 适度增加家庭友好内容（TV-G / TV-Y7 / PG），扩展受众面
4. 电影平均时长约 100 分钟，持续关注观众时长偏好变化

---

**完整可视化看板请打开 `index.html` 查看（10 张高级交互式图表 + 响应式暗黑风格界面）。**